# Phishing Model — Kaggle Training

**Cara pakai:**
1. Upload `emails.csv` sebagai Kaggle Dataset (sekali aja)
2. Add dataset itu ke notebook ini
3. Settings → Internet → **ON**
4. Settings → Accelerator → **GPU T4 x2** (atau P100)
5. Klik **Save Version → Save and Run All** → jalan di background
6. Setelah selesai, download `phishing_v1_checkpoint.zip` dari Output tab

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available! Go to Settings → Accelerator → GPU T4 x2')

In [ ]:
%%capture
!pip install transformers "datasets>=2.14.0,<3.0.0" accelerate evaluate \
    wandb sentencepiece pandas regex scikit-learn pyyaml

In [ ]:
import os

!git clone https://github.com/PJK-GM095-PIJAK/prior-mail-model.git
%cd prior-mail-model
!git checkout feat/phishing-model
!git log --oneline -1

# Add repo to Python path
import sys
sys.path.insert(0, '/kaggle/working/prior-mail-model')

In [ ]:
import shutil, os
from pathlib import Path

# Cari emails.csv dari Kaggle dataset yang sudah di-add
# Ganti nama dataset sesuai yang kamu upload di Kaggle
# Biasanya ada di /kaggle/input/<dataset-name>/emails.csv
possible_paths = list(Path('/kaggle/input').rglob('emails.csv'))

if possible_paths:
    src = possible_paths[0]
    dst = Path('emails.csv')
    shutil.copy(src, dst)
    print(f'Found emails.csv at {src} → copied to {dst}')
else:
    raise FileNotFoundError(
        'emails.csv tidak ditemukan di /kaggle/input/. '
        'Upload emails.csv sebagai Kaggle Dataset dulu, '
        'lalu Add ke notebook ini via + Add Data.'
    )

In [ ]:
# Wandb offline — JANGAN kasih komentar di baris %env ini
%env WANDB_MODE=offline

In [ ]:
!PYTHONPATH=. python -m src.data.prepare --phishing

# Verifikasi splits terbentuk
from datasets import load_from_disk
ds = load_from_disk('data/processed/phishing')
print('Splits:', {k: ds[k].num_rows for k in ds})

In [ ]:
!PYTHONPATH=. python -m src.training.train_phishing \
    --config configs/phishing_v1.yaml

In [ ]:
import os, json
from pathlib import Path

ckpt = Path('checkpoints/phishing_v1')
files = list(ckpt.glob('*'))
print('Checkpoint files:')
for f in sorted(files):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<40} {size_mb:.1f} MB')

# Tampilkan val metrics
val_metrics_path = ckpt / 'val_metrics.json'
if val_metrics_path.exists():
    val = json.loads(val_metrics_path.read_text())
    print()
    print('Val metrics:')
    for k, v in val.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')

In [ ]:
import shutil

# Zip checkpoint → otomatis tersimpan di Output tab Kaggle
output_zip = '/kaggle/working/phishing_v1_checkpoint'
shutil.make_archive(output_zip, 'zip', 'checkpoints/phishing_v1')
print(f'Saved: {output_zip}.zip')
print()
print('Download dari: Kaggle notebook → Output tab → phishing_v1_checkpoint.zip')